# Utilizing Rule-Based Semantic Extraction and Text Parsing to Automate Scoring Cards in *Magic: The Gathering*
### IMT 575 - Spring 2026
Samuel Lee, Sitong Liu, Zach Greenman

In [1]:
import pandas as pd
import numpy as np

from mtg_parser.utils.paths import RAW_DIR, PROCESSED_DIR
from mtg_parser.data.extract_creatures import build_creatures_dataset

In [2]:
build_creatures_dataset()

[pipeline] Default-cards dataset already exists.
[pipeline] Creatures dataset already exists.


In [3]:
# Define features we are interested in
SCRYFALL_COLS = [
    'oracle_id',
    'name',
    'set',
    'released_at',
    'cmc',
    'power',
    'toughness',
    'oracle_text',
    'keywords',
    'color_identity',
    'rarity',
]

POINTS_COLS = ['name', 'set', 'set_name', 'points']

# Load data
creatures_full = pd.read_json(RAW_DIR / 'creatures_full.ndjson', lines=True)

combined_points = pd.read_csv(RAW_DIR / 'combined_csv_pts.csv')

combined_points = combined_points[POINTS_COLS]

# Remove duplicates and ensure earliest version of the card for each set
creatures_full['arena_priority'] = (
    creatures_full['arena_id'].notna().astype(int)
)

creatures_full['released_at'] = pd.to_datetime(
    creatures_full['released_at'], errors='coerce'
)

creatures_full = creatures_full.sort_values(
    by=['arena_priority', 'released_at']
)

creatures_full = creatures_full.drop_duplicates(
    subset=['name', 'set'], keep='first'
)

creatures_subset = creatures_full[SCRYFALL_COLS]

# Merge
merged = combined_points.merge(
    creatures_subset,
    on=['name', 'set'],
    how='left',
)

merged['oracle_text'] = merged['oracle_text'].fillna('')

# Add the release year for each card
merged['year'] = merged['released_at'].dt.year

# Get numeric power/toughness values
merged['power_numeric'] = pd.to_numeric(merged['power'], errors='coerce')

merged['toughness_numeric'] = pd.to_numeric(
    merged['toughness'], errors='coerce'
)

column_order = [
    'oracle_id',
    'name',
    'set',
    'set_name',
    'released_at',
    'year',
    'cmc',
    'power',
    'toughness',
    'power_numeric',
    'toughness_numeric',
    'keywords',
    'oracle_text',
    'color_identity',
    'rarity',
    'points',
]

merged = merged[column_order]

# Diagnostics
missing = merged['oracle_id'].isna().sum()

print(f'[cleaning] Missing oracle_id rows: {missing}')

dupes = merged.duplicated(subset=['name', 'set']).sum()

print(f'[cleaning] Duplicate values in rows: {dupes}')

[cleaning] Missing oracle_id rows: 0
[cleaning] Duplicate values in rows: 0


In [8]:
merged.to_csv(
    PROCESSED_DIR / 'combined_points_cleaned.csv',
    index=False
)

print('Saved combined_points_cleaned.csv to ./data/processed')

Saved combined_points_cleaned.csv to ./data/processed
